In [78]:
pip install torcheval

In [79]:
import torch
import torchvision
import torch.nn.functional as F
from torch import nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torcheval.metrics.functional import multiclass_f1_score, multiclass_precision, multiclass_recall, r2_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold, KFold

In [80]:
df = pd.read_excel("Dry_Bean_Dataset.xlsx")

In [81]:
X = df.drop(['Class'], axis=1)
y = df[['Class']]

In [82]:
le = LabelEncoder()
y_encoded = le.fit_transform(df['Class'].values)

num_classes = len(le.classes_)
num_features = X.shape[1]

In [83]:
X_tensor = torch.tensor(X.values).float()
y_tensor = torch.tensor(y_encoded).long()

In [84]:
class SimpleClassificationModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SimpleClassificationModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

model = SimpleClassificationModel(num_features, num_classes)

In [85]:
num_epochs = 100
lr = 0.001

In [86]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_f1 = []
all_precision = []
all_recall = []

In [87]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_tensor, y_tensor)):
    X_train, X_val = X_tensor[train_idx], X_tensor[val_idx]
    y_train, y_val = y_tensor[train_idx], y_tensor[val_idx]

    scaler = StandardScaler()
    X_train = torch.tensor(scaler.fit_transform(X_train)).float()
    X_val = torch.tensor(scaler.transform(X_val)).float()

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)

    model = SimpleClassificationModel(num_features, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr)

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, predicted = torch.max(outputs.data, 1)

    f1 = multiclass_f1_score(predicted, y_val, num_classes=num_classes, average="macro")
    precision = multiclass_precision(predicted, y_val, num_classes=num_classes, average='macro')
    recall = multiclass_recall(predicted, y_val, num_classes=num_classes, average='macro')

    all_f1.append(f1.item())
    all_precision.append(precision.item())
    all_recall.append(recall.item())

print(f"\nAverage Precision: {np.mean(all_precision):.4f}")
print(f"Average Recall: {np.mean(all_recall):.4f}")
print(f"Average F1: {np.mean(all_f1):.4f}")


Average Precision: 0.9406
Average Recall: 0.9385
Average F1: 0.9392


In [88]:
# Второй датасет

In [89]:
df = pd.read_excel("Concrete_Data.xls")

In [90]:
df.columns = df.columns.str.strip()
X = df.drop(['Concrete compressive strength(MPa, megapascals)'], axis=1)
y = df[['Concrete compressive strength(MPa, megapascals)']]

In [91]:
num_features = X.shape[1]

In [92]:
X_tensor = torch.tensor(X.values).float()
y_tensor = torch.tensor(y.values).float().reshape(-1, 1)

In [93]:
class SimpleRegressionModel(nn.Module):
    def __init__(self, input_size):
        super(SimpleRegressionModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

model = SimpleRegressionModel(num_features)

In [94]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_r2 = []
all_mse = []

In [95]:
num_epochs = 100
lr = 0.001

In [96]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X_tensor)):
    X_train, X_val = X_tensor[train_idx], X_tensor[val_idx]
    y_train, y_val = y_tensor[train_idx], y_tensor[val_idx]

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train = torch.tensor(scaler_X.fit_transform(X_train)).float()
    X_val = torch.tensor(scaler_X.transform(X_val)).float()
    y_train = torch.tensor(scaler_y.fit_transform(y_train)).float()
    y_val = torch.tensor(scaler_y.transform(y_val)).float()

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)

    model = SimpleRegressionModel(num_features)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr)

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)

    r2 = r2_score(outputs, y_val)
    mse = mean_squared_error(outputs, y_val)

    all_r2.append(r2.item())
    all_mse.append(mse.item())

print(f"\nAverage R2: {np.mean(all_r2):.4f}")
print(f"Average MSE: {np.mean(all_mse):.4f}")


Average R2: 0.9044
Average MSE: 0.0950
